# How Much Accuracy Do You Need?

Original QMCPy demo: [`QMCPy/demos/demo_resume_data/accuracy_and_resume.ipynb`](../../../QMCPy/demos/demo_resume_data/accuracy_and_resume.ipynb)

Sou-Cheng Choi (with some edits by Fred Hickernell)

This Julia translation follows the same theme: start with a loose tolerance, then tighten the target only if you need more accuracy.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/demo_resume_data/accuracy_and_resume.ipynb)

Parity note: the QMCPy notebook tightens from `1e-6` to `1e-7`. For this Julia example those targets exceed the current lattice sample window, so the checked-in notebook uses `1e-4` down to `2.5e-5` while preserving the same resume-versus-fresh comparison pattern.


## Art Owen's Reflections on Lyness and the Accuracy Question

The motivating question is the same as in QMCPy: if a first answer is already useful, there is no reason to discard that work when tighter accuracy is requested later.


## The Problem

Automatic quadrature routines require the user to specify a target accuracy. In practice, scientists often do not know that accuracy in advance, or their needs change after they inspect a first result.


## The Solution: Resumable Integration in QMCPy

### How it Works

With resumable integration, the result of an earlier run is not wasted. A second solve at a tighter tolerance can continue from the saved state instead of restarting from zero.


## Implementation

The same implementation ideas used in QMCPy are available in `QMC.jl`: the `integrate` method accepts a `resume` argument, and the stopping criteria can record an iteration log when `trace_iterations=true`.


In [1]:
using QMC
using Serialization
using Printf

function demo_table(rows, columns; caption="")
    io = IOBuffer()
    print(io, "<table>")
    isempty(caption) || print(io, "<caption>", caption, "</caption>")
    print(io, "<thead><tr>")
    foreach(column -> print(io, "<th>", column[2], "</th>"), columns)
    print(io, "</tr></thead><tbody>")
    for row in rows
        print(io, "<tr>")
        foreach(column -> print(io, "<td style=\"text-align:right\">", column[3](getproperty(row, column[1])), "</td>"), columns)
        print(io, "</tr>")
    end
    print(io, "</tbody></table>")
    return Base.HTML(String(take!(io)))
end

format_seconds(x) = abs(x) < 1e-3 ? @sprintf("%.3e", x) : @sprintf("%.3f", x)

summary_columns = [
    (:abs_tol, "abs tol", x -> @sprintf("%.3e", x)),
    (:n_total, "n total", string),
    (:time, "time (s)", format_seconds),
    (:solution, "solution", x -> @sprintf("%.8g", x)),
]


4-element Vector{Tuple{Symbol, String, Function}}:
 (:abs_tol, "abs tol", var"#10#11"())
 (:n_total, "n total", string)
 (:time, "time (s)", Main.format_seconds)
 (:solution, "solution", var"#12#13"())

### Step 1: Quick Estimate

Suppose you want a quick answer, so you set a loose tolerance. We use a Genz oscillatory benchmark and a fixed seed to keep the notebook reproducible.


In [2]:
function make_solver(abs_tol; seed=7, dimension=3)
    dd = Lattice(dimension; seed=seed)
    f = Genz(Uniform(dd); kind=:oscillatory, a=ones(dimension), u=0.5 .* ones(dimension))
    return CubQMCLatticeG(f; abs_tol=abs_tol, trace_iterations=true)
end

loose_tol = 1e-4
result_loose = integrate(make_solver(loose_tol))
@printf("Loose solve: solution = %.8f, n_total = %d, time = %.4f s
", result_loose.solution, result_loose.data[:n_total], result_loose.data[:time_integrate])
sort!(collect(keys(result_loose.data)))


Loose solve: solution = -0.06235934, n_total = 65536, time = 0.0317 s


9-element Vector{Symbol}:
 :converged
 :error_bound
 :iteration_log
 :n
 :n_iterations
 :n_per_rep
 :n_reps
 :n_total
 :time_integrate

The returned data object contains the estimated solution, the error bound, the number of samples used, the iteration log, and the measured solve time.


### Step 2: Save the State (Optional)

You can save the integration state to disk and resume later, or keep the state in memory and continue immediately.

# Save compressed - automatically creates 'data.pkl.gz'

QMCPy's example writes compressed Python pickle files at this point. The Julia notebook instead saves the resumable state as a `.jls` artifact in the local `output/` directory.

# Load compressed - auto-detection works

Resuming in `QMC.jl` likewise uses the serialized `.jls` state directly rather than Python's pickle auto-detection path.


In [3]:
output_dir = joinpath(@__DIR__, "output")
mkpath(output_dir)
resume_path = joinpath(output_dir, "accuracy_and_resume_loose.jls")
serialize(resume_path, result_loose.data)
println("Saved state to: ", resume_path)


Saved state to: /Users/terrya/Documents/ProgramData/QMCSoftware_space/QMC.jl/demos/demo_resume_data/output/accuracy_and_resume_loose.jls

### Step 3: Resume with Tighter Tolerance

Now assume you decide that the first answer is not accurate enough. Resume from the saved state and ask for a tighter tolerance.


In [4]:
tight_tol = 2.5e-5
result_resume = integrate(make_solver(tight_tol); resume=deserialize(resume_path))
@printf("Resumed solve: solution = %.8f, n_total = %d, incremental time = %.4f s
", result_resume.solution, result_resume.data[:n_total], result_resume.data[:time_integrate])


Resumed solve: solution = -0.06236067, n_total = 262144, incremental time = 0.1383 s


### Step 4: Compare to Starting from Scratch

To see what the resume feature saved, solve the same tight problem again from scratch.


In [5]:
result_fresh = integrate(make_solver(tight_tol))
@printf("Fresh tight solve: solution = %.8f, n_total = %d, time = %.4f s
", result_fresh.solution, result_fresh.data[:n_total], result_fresh.data[:time_integrate])
@printf("Saved samples: %d
", result_fresh.data[:n_total] - result_loose.data[:n_total])
@printf("Resumed vs fresh difference: %.3e
", abs(result_resume.solution - result_fresh.solution))

@assert abs(result_resume.solution - result_fresh.solution) ≤ result_resume.data[:error_bound] + result_fresh.data[:error_bound]
@assert result_resume.data[:n_total] == result_fresh.data[:n_total]


Fresh tight solve: solution = -0.06236067, n_total = 262144, time = 0.0343 s
Saved samples: 196608
Resumed vs fresh difference: 0.000e+00


**Step 5: What If You Tighten the Tolerance More Than Once?**

The benefit of resumption is clearer when you solve a whole sequence of tighter tolerances. The fresh workflow repeats work at every tolerance, while the resumed workflow only adds the extra work needed for the next target.


In [6]:
tols = [1e-4, 7.5e-5, 5e-5, 2.5e-5]

function compare_tolerance_sequence(tols)
    fresh_rows = NamedTuple[]
    for eps in tols
        r = integrate(make_solver(eps))
        push!(fresh_rows, (abs_tol=eps, n_total=r.data[:n_total], time=r.data[:time_integrate], solution=r.solution))
    end

    resume_rows = NamedTuple[]
    resume_state = nothing
    for eps in tols
        r = isnothing(resume_state) ? integrate(make_solver(eps)) : integrate(make_solver(eps); resume=resume_state)
        push!(resume_rows, (abs_tol=eps, n_total=r.data[:n_total], time=r.data[:time_integrate], solution=r.solution))
        resume_state = r.data
    end
    return fresh_rows, resume_rows
end

fresh_rows, resume_rows = compare_tolerance_sequence(tols)
display(demo_table(fresh_rows, summary_columns; caption="Fresh solves"))
display(demo_table(resume_rows, summary_columns; caption="Resumed solves"))

fresh_total_samples = sum(row.n_total for row in fresh_rows)
resume_total_samples = resume_rows[end].n_total
println("Total fresh samples across all solves: ", fresh_total_samples)
println("Samples after chained resumption: ", resume_total_samples)

@assert resume_total_samples ≤ fresh_total_samples
@assert all(abs(f.solution - r.solution) ≤ f.abs_tol for (f, r) in zip(fresh_rows, resume_rows))


abs tol,n total,time (s),solution
1.000e-04,65536,0.011,-0.062359339
7.500e-05,65536,0.041,-0.062359339
5.000e-05,131072,0.011,-0.062361949
2.500e-05,262144,0.021,-0.062360665


abs tol,n total,time (s),solution
1.000e-04,65536,0.051,-0.062359339
7.500e-05,131072,0.058,-0.062361949
5.000e-05,262144,0.082,-0.062360665
2.500e-05,524288,0.127,-0.062358812


Total fresh samples across all solves: 524288
Samples after chained resumption: 524288


## Conclusion

The shared lesson from the QMCPy notebook carries over directly: ask for loose accuracy first, inspect the answer, and only pay for tighter tolerances when the application actually needs them.
